<a href="https://colab.research.google.com/github/nivethithanm/torchcode-solutions/blob/main/FIX_08_kv_cache.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FIX-08: KV Cache — Why It Exists and How It Works

**Your issue**: 'KV Cache is completely unintuitive.'

**The core problem**: When generating text token by token, we recompute attention over ALL previous tokens every step. That's wasteful — the K and V for previous tokens haven't changed!

```
Step 1: Generate token 1 → compute K,V for token 1
Step 2: Generate token 2 → recompute K,V for token 1 (wasteful!!) + compute K,V for token 2
Step 3: Generate token 3 → recompute K,V for 1,2 (wasteful!!) + compute K,V for token 3
```

**KV Cache solution**: Save K,V from previous steps. Only compute K,V for the NEW token.

```
Step 1: Compute K1,V1, cache them. Q1 attends to K1,V1.
Step 2: Compute K2,V2, append to cache. Q2 attends to [K1,K2],[V1,V2].
Step 3: Compute K3,V3, append to cache. Q3 attends to [K1,K2,K3],[V1,V2,V3].
```

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## Part 1: See the waste without KV cache

In [2]:
# Simulate generating 5 tokens without caching
D = 8  # tiny for clarity

W_q = torch.randn(D, D)
W_k = torch.randn(D, D)
W_v = torch.randn(D, D)

# Pretend these are our token embeddings
tokens = [torch.randn(1, D) for _ in range(5)]

total_ops = 0
for step in range(1, 6):
    # At each step, we have tokens[0..step-1]
    all_tokens = torch.cat(tokens[:step], dim=0)  # (step, D)

    # Compute Q, K, V for ALL tokens (wasteful!)
    Q = all_tokens @ W_q  # (step, D)
    K = all_tokens @ W_k  # (step, D)
    V = all_tokens @ W_v  # (step, D)

    # We only NEED Q for the last token, K/V for all
    # But we recomputed K/V for previous tokens unnecessarily
    ops_this_step = step  # number of K,V projections computed
    total_ops += ops_this_step
    print(f'Step {step}: computed {ops_this_step} K,V projections (only 1 is new!)')

print(f'\nTotal K,V projections: {total_ops}')
print(f'With caching: only {5} (one per step)')
print(f'Wasted: {total_ops - 5} projections ({(total_ops-5)/total_ops*100:.0f}% waste!)')

Step 1: computed 1 K,V projections (only 1 is new!)
Step 2: computed 2 K,V projections (only 1 is new!)
Step 3: computed 3 K,V projections (only 1 is new!)
Step 4: computed 4 K,V projections (only 1 is new!)
Step 5: computed 5 K,V projections (only 1 is new!)

Total K,V projections: 15
With caching: only 5 (one per step)
Wasted: 10 projections (67% waste!)


## Part 2: Add KV cache — simplest possible version

In [3]:
# Same setup, but now we CACHE K and V

k_cache = []  # stores K for each token
v_cache = []  # stores V for each token

generated_tokens = []

for step in range(5):
    new_token = tokens[step]  # (1, D)

    # Only compute Q, K, V for the NEW token
    q_new = new_token @ W_q  # (1, D)
    k_new = new_token @ W_k  # (1, D)
    v_new = new_token @ W_v  # (1, D)

    # Append to cache
    k_cache.append(k_new)
    v_cache.append(v_new)

    # Get ALL keys and values from cache
    K_all = torch.cat(k_cache, dim=0)  # (step+1, D)
    V_all = torch.cat(v_cache, dim=0)  # (step+1, D)

    # Attention: new query attends to ALL cached keys/values
    scores = (q_new @ K_all.T) / math.sqrt(D)  # (1, step+1)
    weights = F.softmax(scores, dim=-1)          # (1, step+1)
    output = weights @ V_all                      # (1, D)

    print(f'Step {step}: Q shape (1,{D}), K_cache ({len(k_cache)},{D}), scores (1,{len(k_cache)})')

print('\n✅ Only 1 K,V projection per step!')
print('Cache grows by one row each step.')

Step 0: Q shape (1,8), K_cache (1,8), scores (1,1)
Step 1: Q shape (1,8), K_cache (2,8), scores (1,2)
Step 2: Q shape (1,8), K_cache (3,8), scores (1,3)
Step 3: Q shape (1,8), K_cache (4,8), scores (1,4)
Step 4: Q shape (1,8), K_cache (5,8), scores (1,5)

✅ Only 1 K,V projection per step!
Cache grows by one row each step.


## Part 3: As a class

Now let's make this proper — the pattern used in vLLM/HuggingFace.

In [11]:
class AttentionWithKVCache(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = math.sqrt(self.d_head)

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, kv_cache=None):
        """Forward with optional KV cache.
        Args:
            x: (B, N_new, D) — during generation, N_new=1 (just the new token)
            kv_cache: tuple (cached_K, cached_V) or None
        Returns: (output, new_kv_cache)
        """
        B, N_new, D = x.shape

        # Project ONLY the new tokens
        Q = self.W_q(x)  # (B, N_new, D)
        K_new = self.W_k(x)  # (B, N_new, D)
        V_new = self.W_v(x)  # (B, N_new, D)

        # Append to cache
        if kv_cache is not None:
            K_prev, V_prev = kv_cache
            K = torch.cat([K_prev, K_new], dim=1)  # (B, N_prev + N_new, D)
            V = torch.cat([V_prev, V_new], dim=1)
        else:
            K = K_new
            V = V_new

        # Store updated cache
        new_cache = (K, V)

        # Reshape for multi-head
        def split(t):
            B, N, _ = t.shape
            return t.view(B, N, self.n_heads, self.d_head).transpose(1, 2)

        Q_h = split(Q)  # (B, H, N_new, d_head)
        K_h = split(K)  # (B, H, N_total, d_head)  ← includes cache!
        V_h = split(V)  # (B, H, N_total, d_head)

        # Attention: Q is small (N_new=1), K/V are big (N_total)
        scores = Q_h @ K_h.transpose(-2, -1) / self.scale  # (B, H, 1, N_total)
        weights = F.softmax(scores, dim=-1)
        attn_out = weights @ V_h  # (B, H, 1, d_head)

        # Merge heads
        B, H, N, d = attn_out.shape
        merged = attn_out.transpose(1, 2).contiguous().view(B, N, H * d)

        return self.W_o(merged), new_cache

# Test: simulate generating 5 tokens one at a time
attn = AttentionWithKVCache(d_model=64, n_heads=4)

# Prefill: process the prompt (3 tokens at once)
prompt = torch.randn(1, 3, 64)
out, cache = attn(prompt, kv_cache=None)
print(f'Prefill: input (1,3,64) → output {out.shape}, cache K {cache[0].shape}')

# Decode: generate tokens one at a time
for step in range(4):
    new_token = torch.randn(1, 1, 64)  # ONE new token
    out, cache = attn(new_token, kv_cache=cache)
    print(f'Decode step {step}: input (1,1,64) → output {out.shape}, cache K {cache[0].shape}')

print('\n✅ KV Cache: cache grows by 1 each step, Q is always size 1 during decode')

Prefill: input (1,3,64) → output torch.Size([1, 3, 64]), cache K torch.Size([1, 3, 64])
Decode step 0: input (1,1,64) → output torch.Size([1, 1, 64]), cache K torch.Size([1, 4, 64])
Decode step 1: input (1,1,64) → output torch.Size([1, 1, 64]), cache K torch.Size([1, 5, 64])
Decode step 2: input (1,1,64) → output torch.Size([1, 1, 64]), cache K torch.Size([1, 6, 64])
Decode step 3: input (1,1,64) → output torch.Size([1, 1, 64]), cache K torch.Size([1, 7, 64])

✅ KV Cache: cache grows by 1 each step, Q is always size 1 during decode


## Key takeaway

```
WITHOUT cache:  Q=(B,N_total,D), K=(B,N_total,D)  → recompute ALL K,V every step
WITH cache:     Q=(B,1,D),       K=(B,N_total,D)  → only compute K,V for new token
```

The cache stores K and V from previous steps. Each decode step:
1. Compute Q, K_new, V_new for the ONE new token
2. Append K_new, V_new to cache
3. Attention: Q (size 1) attends to full cache (size grows)

This is why **decode is memory-bound**: the Q is tiny (1 token) but K,V cache can be huge.